# SAM3D Objects: mask fit and LiDAR free space (pilot)

**Question.** SAM3D Objects gets a (very good) SAM3 mask and a pointmap whose depth is calibrated on the LiDAR. Does its mesh
then (1) fit the mask, (2) sit on the LiDAR depth, and (3) stay out of the space the LiDAR has certified as empty?
Phase 0 saw OBBs reach into free space; here we look at the **meshes themselves** and check the OBB and the GT box next to them.

Nothing is assumed in advance. What is measured, per object:

| step | measurement | how to read it |
|---|---|---|
| 5 | **Mask fit**: the full mesh is rendered with the camera intrinsics; recall (mask pixels the mesh misses), leak (mesh pixels outside the mask, split into *behind another detection* = explainable and *unexplained*), IoU | recall < 1 or a big unexplained leak = the mesh does not follow the mask |
| 6 | **Depth on the mesh**: LiDAR return depth minus rendered mesh depth at the same pixel | > 0: mesh surface nearer than the measured surface; < 0: return is in front of the mesh |
| 7 | **Free space**: share of the mesh surface / filled mesh volume / fitted OBB / GT box that lies in space certified free by the keyframe's LiDAR sweep, counting only voxels at least `D_CLAIM` m in front of the nearest return | the **GT box column is the baseline**: a correct box also holds some free air (corners, gaps under the car). Only the excess over it is overshoot |
| 8 | **Mask perturbation** (optional, GPU): re-run SAM3D with an eroded / dilated / shifted / box-filled mask | mask is a constraint: the mesh follows the *new* mask (IoU-new high, IoU-orig drops). A hint: the mesh stays put |

**Timing.** The camera and the LiDAR sweep are not simultaneous (printed per frame). The dataset is ego-motion compensated, but an object that
moves itself can still be displaced by the offset, so every object carries its GT speed (`motion` column) and static objects (speed ~ 0) are the clean cases; where the dataset has no velocity (`unknown`) judge from the images.

Free space = voxels a beam passed through (one traversal is enough, single sweep); occupied = a beam ended there; unknown = the rest. Occupancy is
computed from the **full single sweep including ground** (ground returns certify the free air above the ground).

## 1. Setup

In [ ]:
import sys, gc, os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from IPython.display import display

DEV_ROOT = Path('/workspace')
sys.path.insert(0, str(DEV_ROOT / 'autolabeling' / 'src'))
from autolabeling import pilot
from autolabeling.utils import freespace as fs, freespace_viz as fv, mesh_mask as mm
from autolabeling.utils.diagnostics import associate_gt_lidar, resolve_duplicate_gt, object_errors
from autolabeling.utils.geometry import cam_to_ego

# ── What to run ──────────────────────────────────────────────────────────────────
print('pilot frames:', [s['tag'] for s in pilot.SELECTIONS])
RUN_TAGS = [s['tag'] for s in pilot.SELECTIONS]   # all frames at once; or e.g. ['ecp10_f560'] to work scene by scene (finished stages are cached, so re-running is cheap)
SHOW_OBJECT_FIGS = 3                    # per frame: how many per-object figures to DISPLAY (all are saved as PNG in the results folder)
SELECTIONS = [s for s in pilot.SELECTIONS if s['tag'] in RUN_TAGS]
# `tag` names the checkpoint folder of a frame: change the tag if you change scene / frame / camera of a selection.
POINTMAP_MODE = 2                       # moge_affine_local = the current default
VOXEL = 0.1                             # free-space grid resolution [m]
MARGINS = (0.0, 0.1, 0.2, 0.3)          # a mesh point only counts as violating if it is this far in front of the nearest return
D_CLAIM = 0.2                           # the margin used for the headline numbers
MIN_STATIC_SPEED = 0.5                  # m/s: GT speed below this = "static"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_ROOT = DEV_ROOT / 'autolabeling' / 'output' / 'freespace_pilot'
RESULT_DIR = DEV_ROOT / 'results' / 'sam3d_mask_freespace'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print('device:', DEVICE, '| checkpoints:', CKPT_ROOT)

## 2. Load the frames

Image, the keyframe's LiDAR sweep (ground included), GT boxes with their speed, and the camera/LiDAR time offset.

In [ ]:
NUSC, CFG = globals().get('NUSC', {}), globals().get('CFG', {})     # kept between re-runs: opening the ECP annotations takes minutes
CTX = {}
for sel in SELECTIONS:
    ds, tag = sel['dataset'], sel['tag']
    if ds not in NUSC:
        NUSC[ds] = pilot.open_nusc(ds)
        CFG[ds] = pilot.load_cfg(ds, POINTMAP_MODE)
    nusc = NUSC[ds]
    frame = pilot.get_frame(nusc, sel)
    img, _ = frame.load_images()
    pts, sensor = pilot.single_sweep(nusc, frame)
    CTX[tag] = dict(sel=sel, nusc=nusc, frame=frame, img=img, H=img.shape[0], W=img.shape[1], pts=pts, sensor=sensor,
                    timing=pilot.sweep_timing(nusc, frame), gts=pilot.gt_with_speed(nusc, frame),
                    gt_cats=pilot.DATASET_INFO[ds]['gt_categories'])
    print(f"{tag:12s} {frame.scene_name}  {sel['camera']:16s} {img.shape[1]}x{img.shape[0]}  LiDAR points {len(pts):6d}  "
          f"GT boxes {len(CTX[tag]['gts']):3d}  camera-LiDAR offset {CTX[tag]['timing']['camera_minus_lidar_ms']:+.1f} ms")

fig, axes = plt.subplots(1, len(CTX), figsize=(5 * len(CTX), 3.2), squeeze=False)
for ax, (tag, c) in zip(axes[0], CTX.items()):
    ax.imshow(c['img']); ax.axis('off'); ax.set_title(tag, fontsize=9)
plt.tight_layout()

## 3. Run the pipeline as is (SAM3 -> TerraSeg -> SAM3D Objects)

The production pipeline, one frame at a time, with two changes: full meshes with faces are kept (`mesh_points: 0`) and the pedestrian prompt is removed
(SAM3D Body is not analysed here; bicycles therefore keep their own OBB, no rider merge). Finished stages are cached under the checkpoint folder,
so re-running the cell only recomputes what is missing. **Needs the GPU.**

In [ ]:
for tag, c in CTX.items():
    print('=' * 100); print(tag)
    frame, objs = pilot.run_selection(c['sel'], c['nusc'], CFG[c['sel']['dataset']], CKPT_ROOT, DEVICE)
    c['objs'] = objs
    gc.collect(); torch.cuda.empty_cache()
    print(f"  -> {len(objs)} object(s): " + ', '.join(f"#{i+1} {r['prompt']}" for i, r in enumerate(objs)))

### 3.1 Detections (numbers = object ids used everywhere below)

In [ ]:
COLORS = {'car': (80, 160, 255), 'bicycle': (80, 220, 120), 'motorcycle': (255, 200, 60), 'bus': (200, 100, 255),
          'truck': (255, 140, 60), 'construction vehicle': (255, 90, 160), 'trailer': (120, 220, 220)}
fig, axes = plt.subplots(1, len(CTX), figsize=(7 * len(CTX), 4.5), squeeze=False)
for ax, (tag, c) in zip(axes[0], CTX.items()):
    ov = c['img'].astype(np.float32).copy()
    for r in c['objs']:
        m = r['binary_mask']; ov[m] = 0.5 * ov[m] + 0.5 * np.array(COLORS.get(r['prompt'], (200, 200, 200)))
    ax.imshow(ov.astype(np.uint8)); ax.axis('off'); ax.set_title(tag, fontsize=9)
    for i, r in enumerate(c['objs']):
        ys, xs = np.where(r['binary_mask'])
        ax.text(xs.mean(), ys.mean(), str(i + 1), color='w', fontsize=10, fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.2', fc='k', ec='w', lw=1))
plt.tight_layout()
rows = [dict(tag=tag, id=i + 1, cls=r['prompt'], score=round(float(r['score']), 2), mask_px=int(r['binary_mask'].sum()),
             pointmap_fit=r.get('o3_mode'), mesh_verts=len(r['vertices']), faces=None if r['faces'] is None else len(r['faces']))
        for tag, c in CTX.items() for i, r in enumerate(c['objs'])]
display(pd.DataFrame(rows))

## 4. Match each object to its GT box (and get its speed)

Same LiDAR-based rule as the earlier diagnostics: the GT box that contains most of the single-sweep returns that fall on the mask. Objects without a match
are still analysed (mask fit, free space of mesh and OBB) but have no GT baseline.

In [ ]:
for tag, c in CTX.items():
    claims = {}
    for i, r in enumerate(c['objs']):
        gi, frac, amb = associate_gt_lidar(c['gts'], r['binary_mask'], c['frame'], c['pts'], category_prefix=c['gt_cats'])
        r['gt_idx'] = None if (gi is None or amb) else gi
        if r['gt_idx'] is not None:
            claims[i] = (gi, int(r['binary_mask'].sum()))
    for i in resolve_duplicate_gt(claims):
        c['objs'][i]['gt_idx'] = None
rows = []
for tag, c in CTX.items():
    for i, r in enumerate(c['objs']):
        g = None if r['gt_idx'] is None else c['gts'][r['gt_idx']]
        row = dict(tag=tag, id=i + 1, cls=r['prompt'], gt=None if g is None else g['category'],
                   gt_speed=None if g is None else round(g['speed'], 2),
                   gt_lidar_pts=None if g is None else g['num_lidar_pts'])
        if g is not None:
            row.update({k: round(v, 2) for k, v in object_errors(r['obb_center'], r['obb_dims'], r['obb_yaw'], g).items()
                        if k in ('center_err', 'long_ratio', 'short_ratio', 'h_ratio', 'yaw_err', 'gt_range')})
        rows.append(row)
display(pd.DataFrame(rows))

### 4.1 Calibration check: GT boxes and OBBs drawn into the image

Green = GT boxes, orange = OBBs of the SAM3D meshes (numbered). They use exactly the intrinsics / extrinsics that the mesh rendering and the LiDAR projection use.
**If the green GT boxes do not sit on the real objects here, the camera-LiDAR calibration is off and every mask-vs-mesh number below has to be read with that in mind.**

In [ ]:
for tag, c in CTX.items():
    fig, ax = plt.subplots(figsize=(15, 8.5))
    fv.boxes_on_image(ax, c['img'], c['frame'], [g['corners_3d'] for g in c['gts']],
                      [dict(corners=r['obb_corners'], label=i + 1) for i, r in enumerate(c['objs'])], title=f'{tag}: GT boxes (green) and mesh OBBs (orange) in the image')
    plt.tight_layout(); plt.show()

## 5. Free-space grid and the occupancy map of each frame

Green = certified free, grey = unknown, black = occupied (a beam ended there). White dots: LiDAR returns between 0.3 and 3 m height.
Dashed green boxes = GT, coloured boxes = OBB of each SAM3D mesh (with its id). The red triangle is the LiDAR.

In [ ]:
for tag, c in CTX.items():
    c['grid'] = fs.build_grid(c['pts'], c['sensor'], vs=VOXEL)
    st = c['grid'].state_grid()
    print(f"{tag:12s} grid {c['grid'].shape}  free {100*(st==fs.FREE).mean():.1f}%  occupied {100*(st==fs.OCC).mean():.2f}%")
fig, axes = plt.subplots(1, len(CTX), figsize=(7 * len(CTX), 7), squeeze=False)
for ax, (tag, c) in zip(axes[0], CTX.items()):
    fv.scene_bev(ax, c['grid'], c['pts'], [g['corners_3d'] for g in c['gts']],
                 [dict(corners=r['obb_corners'], color='orange', label=i + 1) for i, r in enumerate(c['objs'])])
    ax.set_title(tag, fontsize=9)
plt.tight_layout()

## 6. Measure every object

For each object: mesh vs mask (render), depth residuals, and free space of mesh surface, filled mesh volume, OBB volume and GT-box volume.
The filled mesh volume needs a closed shell; where the mesh is open or thin only the surface voxels remain (an underestimate of the volume).
Occluders for the leak split = the masks of the *other detected objects* in the frame (pedestrians are not segmented in this run).

In [ ]:
ROWS, RES = [], {}
for tag, c in CTX.items():
    H, W = c['H'], c['W']
    for i, r in enumerate(c['objs']):
        occ = pilot.occluder_mask([o['binary_mask'] for j, o in enumerate(c['objs']) if j != i], (H, W))
        gt = None if r['gt_idx'] is None else c['gts'][r['gt_idx']]
        row = pilot.analyze_object(r, c['frame'], c['grid'], c['pts'], H, W, gt=gt, occluders=occ, margins=MARGINS, device=DEVICE, claim=D_CLAIM)
        V_ego = row.pop('_verts_ego')
        ctrl = pilot.analyze_controls(r, c['frame'], c['grid'], c['pts'], H, W, V_ego, r['faces'], row['_surf'], row['_vol'], gt=gt, claim=D_CLAIM)
        RES[(tag, i + 1)] = {k: row.pop(k) for k in ('_render', '_surf', '_vol')}
        RES[(tag, i + 1)]['occ'] = occ
        RES[(tag, i + 1)]['curves'] = ctrl.pop('_curves')
        row.update(ctrl)
        row.update(tag=tag, id=i + 1, cls=r['prompt'], has_gt=gt is not None)
        ROWS.append(row)
        print(f"{tag} #{i+1} {r['prompt']}: done")
df = pd.DataFrame(ROWS)
for col in ('gt_speed', 'mesh_vol_outside_gt_box'):
    if col not in df:
        df[col] = np.nan
# static: GT speed below the threshold. NaN speed (no velocity for that annotation) = unknown, not static.
df['motion'] = np.where(df.gt_speed.isna(), 'unknown', np.where(df.gt_speed < MIN_STATIC_SPEED, 'static', 'moving'))

### 6.1 Mesh vs mask

In [ ]:
pd.set_option('display.float_format', lambda v: '%.3f' % v); pd.set_option('display.width', 250)
cols = ['tag', 'id', 'cls', 'mask_mask_px', 'mask_mesh_px', 'mask_recall', 'mask_leak', 'mask_leak_free', 'mask_iou', 'mask_touches_border']
display(df[cols].rename(columns=lambda c: c.replace('mask_', '')))
print('recall = share of mask pixels covered by the mesh;  leak = share of mesh pixels outside the mask;  leak_free = leak NOT behind another detection.')

### 6.2 Mesh depth vs LiDAR depth (returns inside the mask)

In [ ]:
cols = ['tag', 'id', 'cls', 'depth_n_in_mask', 'depth_n_uncovered', 'depth_median', 'depth_p25', 'depth_p75', 'depth_frac_within_0p3']
display(df[cols].rename(columns=lambda c: c.replace('depth_', '')))
print('residual = LiDAR depth - mesh depth [m]; > 0: mesh surface is nearer to the camera than the measured surface. '
      'In-mask returns include some background bleeding through gaps, so read the median / IQR, not single values.')

### 6.3 Free space of the mesh (headline) and of OBB / GT box (secondary)

**Headline = the mesh surface.** `surf_free` = share of 20,000 samples on the mesh surface that lie in certified-free space at least `D_CLAIM` m in front of the nearest return
(the red points in the pictures). `surf_unknown` = share in unknown space: **an object the LiDAR barely saw (hidden behind another object) has a large unknown share and its `surf_free`
says nothing** (no evidence is not "no violation"). `pen_p50 / pen_p90` = how deep the violating points sit in front of the nearest return [m] (penetration depth).
`below_ground` = how far the lowest mesh points (0.5th percentile) lie below the ground next to the object, estimated from the sweep's returns around the object (positive = below; NaN when the ring shows no clear ground level); the same for the OBB and the GT box, whose value is the baseline for the estimate itself. `below_gt_bottom_mesh` needs no ground estimate: mesh bottom minus GT-box bottom (GT boxes rest on the ground; needs a GT match).

**Secondary = boxes and volumes.** `obb_*` / `gt_*` are free volume inside the OBB and the GT box. Compare mesh and box only with care: a car mesh fills less of its bounding box than the box's free corners do, so
"mesh minus GT box" is biased towards finding **less** overshoot than there is. Use the GT box column to see what a correct box holds, not as a subtraction.

In [ ]:
d = f'{D_CLAIM:g}'
def col(name):
    return df[name] if name in df else pd.Series(np.nan, index=df.index)
t = df[['tag', 'id', 'cls', 'has_gt', 'motion']].copy()
t['surf_free'] = col(f'surf_frac_free_m{d}')
t['surf_free_any'] = col('surf_frac_free')
t['surf_unknown'] = col('surf_frac_unknown')
t['pen_p50'] = col('surf_viol_depth_p50'); t['pen_p90'] = col('surf_viol_depth_p90')
t['below_gt_bottom_mesh'] = col('mesh_below_gt_bottom'); t['below_ground_mesh'] = col('mesh_below_ground'); t['below_ground_obb'] = col('obb_below_ground'); t['below_ground_gt'] = col('gt_below_ground')
t['mesh_frac'] = col(f'vol_frac_free_m{d}'); t['obb_frac'] = col(f'obb_frac_free_m{d}'); t['gt_frac'] = col(f'gt_frac_free_m{d}')
t['mesh_m3'] = col(f'vol_free_vol_m3_m{d}'); t['obb_m3'] = col(f'obb_free_vol_m3_m{d}'); t['gt_m3'] = col(f'gt_free_vol_m3_m{d}')
t['mesh_vol_m3'] = col('vol_vol_m3'); t['gt_vol_m3'] = col('gt_vol_m3')
t['mesh_outside_gt'] = col('mesh_vol_outside_gt_box')
T = t
display(T)
print('surf_free: share of the mesh surface in free space >= ' + d + ' m in front of a return.  surf_unknown: share in unknown space (no evidence).  pen_*: depth of the violating points [m].')
print('below_ground_*: lowest 0.5% of the mesh / OBB bottom / GT bottom below the ground estimated next to the object [m] (positive = below); below_gt_bottom_mesh: mesh bottom below the GT box bottom [m].')
print('mesh_outside_gt: share of the filled mesh volume outside the GT box enlarged by 0.2 m.')

In [ ]:
o = T.copy()
lab = [f"{r.tag.split('_')[0]}#{r.id} {r.cls}" + {'static': '', 'moving': ' (moving)', 'unknown': ''}[r.motion] for r in o.itertuples()]
x = np.arange(len(o))
fig, axs = plt.subplots(1, 3, figsize=(max(14, 1.0 * len(o) + 8), 4.4))
ms = [(0.0, 'any free'), (0.1, '>= 0.1 m'), (0.2, '>= 0.2 m'), (0.3, '>= 0.3 m')]
w = 0.2
for k, (m_, name) in enumerate(ms):
    axs[0].bar(x + (k - 1.5) * w, col(f'surf_frac_free_m{m_:g}'), w, label=name)
axs[0].plot(x, o.surf_unknown, 'k_', ms=14, label='in unknown space')
axs[0].set_ylabel('share of mesh surface'); axs[0].set_title('mesh surface in free space (by margin)', fontsize=9); axs[0].legend(fontsize=7)
axs[1].bar(x - 0.2, o.pen_p50, 0.4, label='p50'); axs[1].bar(x + 0.2, o.pen_p90, 0.4, label='p90')
axs[1].set_ylabel('m in front of the nearest return'); axs[1].set_title('penetration depth of the violating points', fontsize=9); axs[1].legend(fontsize=7)
axs[2].bar(x - 0.3, o.below_ground_mesh, 0.2, label='mesh (vs ground estimate)', color='tab:red'); axs[2].bar(x - 0.1, o.below_ground_obb, 0.2, label='OBB', color='tab:orange')
axs[2].bar(x + 0.1, o.below_ground_gt, 0.2, label='GT box', color='tab:green'); axs[2].bar(x + 0.3, o.below_gt_bottom_mesh, 0.2, label='mesh bottom vs GT bottom', color='tab:purple')
axs[2].axhline(0, color='k', lw=0.6); axs[2].set_ylabel('m below the local ground'); axs[2].set_title('reach below ground', fontsize=9); axs[2].legend(fontsize=7)
for ax in axs:
    ax.set_xticks(x); ax.set_xticklabels(lab, rotation=60, ha='right', fontsize=7)
plt.tight_layout()
print('Objects with a large unknown share (black ticks) were barely seen by the LiDAR: read their surf_free as "no evidence".')

In [ ]:
# secondary view: free VOLUME of mesh / OBB / GT box (see the caveat above)
m = T[T.has_gt]
if len(m):
    x = np.arange(len(m)); w = 0.27
    fig, axs = plt.subplots(1, 2, figsize=(max(9, 0.9 * len(m) + 4), 3.8))
    lab = [f"{r.tag.split('_')[0]}#{r.id} {r.cls}" for r in m.itertuples()]
    for ax, (cols, ylab) in zip(axs, ((('mesh_frac', 'obb_frac', 'gt_frac'), 'share of volume in free space'), (('mesh_m3', 'obb_m3', 'gt_m3'), 'free volume [m3]'))):
        for k, (cn, colr) in enumerate(zip(cols, ('tab:red', 'tab:orange', 'tab:green'))):
            ax.bar(x + (k - 1) * w, m[cn], w, color=colr, label=cn.split('_')[0] + (' box' if cn.startswith('gt') else ''))
        ax.set_xticks(x); ax.set_xticklabels(lab, rotation=60, ha='right', fontsize=7); ax.set_ylabel(ylab); ax.legend(fontsize=8)
    plt.suptitle('secondary: free volume inside the filled mesh / OBB / GT box', fontsize=9); plt.tight_layout()

### 6.4 Controls: what do the free-space violations mean?

Three checks on the red points (mesh surface in free space at least `D_CLAIM` m in front of a return), because a correct mesh may also show some (glass, wheel arches and the underbody let beams through):

1. **Floor (`floor_free`, `floor_scaled_free`).** The same mesh moved to the GT position and yaw (`floor_free`), and also stretched to the GT size (`floor_scaled_free`), then re-measured. This is what a correctly placed
   mesh of this shape shows. `surf_free` clearly above `floor_free` = the extra comes from where / how big the mesh is, not from its shape. Needs a GT match. (`floor_unknown` = how much of it is unobserved.)
2. **Ray shift (`best_shift_free`, `best_shift_lidar`).** The mesh slid along the LiDAR ray by -0.5 ... +0.5 m (positive = away from the sensor). `best_shift_free` = the shift with the smallest red share (`free_at_best_shift`), `best_shift_lidar` = the shift that
   puts the surface closest to the in-mask returns (`lidar_dist_at_0` -> `lidar_dist_best`, median distance in m). A clear minimum away from 0 = a real offset along the ray, and how much a correction could recover.
3. **What a depth constraint on the mask would have seen (`red_*`, fractions of the red points).** Each red point is projected into the image:
   `red_implied` = inside the mask AND an in-mask LiDAR return within 30 px lies >= 0.2 m behind it, so making the mesh match the in-mask depth already excludes it;
   `red_in_mask_no_return_near` = inside the mask but no return nearby (sparse LiDAR, depth matching would not see it);
   `red_in_mask_returns_not_behind` = returns nearby but none behind the point;
   `red_outside_mask` = the point projects outside the object's own mask (silhouette not respected);
   `red_off_image` = not in the image. `red_end_in_mesh` = share of the red points whose beam ends inside the mesh volume (the beam went through the surface into the object: glass / gaps, not overshoot).

In [ ]:
cc = ['tag', 'id', 'cls', 'has_gt', 'motion']
C = df[cc].copy()
def dcol(name):
    return df[name] if name in df else pd.Series(np.nan, index=df.index)
C['surf_free'] = dcol(f'surf_frac_free_m{D_CLAIM:g}')
for name in ('floor_free', 'floor_scaled_free', 'floor_unknown', 'shift_free_at_0', 'best_shift_free', 'free_at_best_shift', 'best_shift_lidar', 'lidar_dist_at_0', 'lidar_dist_best', 'n_returns_near',
             'red_n', 'red_implied', 'red_in_mask_no_return_near', 'red_in_mask_returns_not_behind', 'red_outside_mask', 'red_off_image', 'red_end_in_mesh'):
    C[name] = dcol(name)
display(C)

k = C[C.red_n > 0]
if len(k):
    lab = [f"{r.tag.split('_')[0]}#{r.id} {r.cls}" for r in k.itertuples()]
    x = np.arange(len(k))
    fig, axs = plt.subplots(1, 2, figsize=(max(12, 0.9 * len(k) + 6), 4.2))
    bottom = np.zeros(len(k))
    for name, colr in (('red_implied', 'tab:green'), ('red_in_mask_returns_not_behind', 'tab:olive'), ('red_in_mask_no_return_near', 'tab:orange'), ('red_outside_mask', 'tab:red'), ('red_off_image', 'tab:gray')):
        axs[0].bar(x, k[name], bottom=bottom, label=name.replace('red_', ''), color=colr); bottom += k[name].values
    axs[0].set_title('where the violating points project (share)', fontsize=9); axs[0].legend(fontsize=7)
    axs[0].set_xticks(x); axs[0].set_xticklabels(lab, rotation=60, ha='right', fontsize=7)
    m = C.loc[k.index]
    axs[1].bar(x - 0.27, m.surf_free, 0.27, label='mesh as generated', color='tab:red'); axs[1].bar(x, m.floor_free, 0.27, label='mesh at GT position + yaw', color='tab:blue')
    axs[1].bar(x + 0.27, m.floor_scaled_free, 0.27, label='... and GT size', color='tab:cyan')
    axs[1].set_title('share of the mesh surface in free space (>= D_CLAIM): generated vs GT-placed', fontsize=9); axs[1].legend(fontsize=7)
    axs[1].set_xticks(x); axs[1].set_xticklabels(lab, rotation=60, ha='right', fontsize=7)
    plt.tight_layout()

## 7. Pictures per object

Left: image with the SAM3 mask contour (cyan) and the mesh silhouette contour (magenta); red tint = mesh pixels outside the mask (not behind another detection),
blue tint = mask pixels the mesh does not cover; wireframes: OBB of the mesh (orange) and GT box (green). Middle: top-down occupancy around the object with mesh surface points (**red = in free space >= 0.2 m in front of the nearest
return**, orange = free but < 0.2 m, blue = unknown space, purple = on an occupied voxel), LiDAR, OBB (orange) and GT box (green). Right: the same in a side view along the LiDAR ray
(sensor on the left): overshoot towards the sensor is a mesh in front of the measured surface.

In [ ]:
FIG_IDS = None            # None = every object; or e.g. {('nusc3_kf3', 1), ('ecp10_f560', 4)}
shown = {}
for row in df.itertuples():
    key = (row.tag, row.id)
    if FIG_IDS is not None and key not in FIG_IDS:
        continue
    c = CTX[row.tag]; r = c['objs'][row.id - 1]; R = RES[key]
    sil, _ = R['_render']
    gt = None if r['gt_idx'] is None else c['gts'][r['gt_idx']]
    boxes = [dict(corners=r['obb_corners'], color='orange', label='OBB of mesh')]
    if gt is not None:
        boxes.append(dict(corners=fv.box_corners(*pilot.gt_box_of(gt)), color='green', label='GT box'))
    ctr = np.asarray(r['obb_center'], float)
    fig, axs = plt.subplots(1, 3, figsize=(21, 5.6))
    px_boxes = [dict(px=fv.project_corners(c['frame'], r['obb_corners']), color='orange')]
    if gt is not None:
        px_boxes.append(dict(px=fv.project_corners(c['frame'], gt['corners_3d']), color='lime'))
    fv.image_overlay(axs[0], c['img'], r['binary_mask'], sil, R['occ'], boxes_px=px_boxes,
                     title=f"{row.tag} #{row.id} {row.cls}   IoU {row.mask_iou:.2f}  recall {row.mask_recall:.2f}  leak {row.mask_leak_free:.2f}")
    fv.bev_panel(axs[1], c['grid'], ctr, R['_surf'], c['pts'], boxes, half=max(6.0, 0.7 * max(r['obb_dims'][:2])),
                 title=f"top view   mesh surface in free space (>= {D_CLAIM} m): {df.loc[row.Index, f'surf_frac_free_m{D_CLAIM:g}']:.2f}, unknown {df.loc[row.Index, 'surf_frac_unknown']:.2f}")
    axs[1].legend(fontsize=7, loc='upper right')
    fv.side_panel(axs[2], c['grid'], ctr, R['_surf'], c['pts'], boxes, half=max(6.0, 0.7 * max(r['obb_dims'][:2])), title='side view along the LiDAR ray')
    axs[2].legend(fontsize=7, loc='upper right')
    plt.tight_layout()
    fig.savefig(RESULT_DIR / f"{row.tag}_obj{row.id}_{row.cls.replace(' ', '_')}.png", dpi=90, bbox_inches='tight')
    if FIG_IDS is not None or shown.get(row.tag, 0) < SHOW_OBJECT_FIGS:
        shown[row.tag] = shown.get(row.tag, 0) + 1
        plt.show()
    else:
        plt.close(fig)

### 7.0 Control figure per object

Left: share of the mesh surface in free space (red) and on occupied voxels (black) as the mesh is slid along the LiDAR ray (positive = away from the sensor), with the GT-placed floors as dashed lines.
Right: median distance from the in-mask LiDAR returns to the mesh surface for the same shifts. A minimum away from 0 = an offset along the ray.

In [ ]:
shown = {}
for row in df.itertuples():
    key = (row.tag, row.id)
    cur = RES[key]['curves']['shift']
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
    axs[0].plot(cur['shifts'], cur['free'], 'o-', color='tab:red', label='in free space (>= D_CLAIM)')
    axs[0].plot(cur['shifts'], cur['occ'], 's-', color='k', label='on occupied voxels')
    for name, ls, lab_ in (('floor_free', '--', 'floor: mesh at GT pose'), ('floor_scaled_free', ':', 'floor: at GT pose and size')):
        v_ = df.loc[row.Index, name] if name in df else np.nan
        if np.isfinite(v_):
            axs[0].axhline(v_, color='tab:blue', ls=ls, label=lab_)
    axs[0].axvline(0, color='gray', lw=0.6); axs[0].set_xlabel('shift along the LiDAR ray [m]  (+ = away from the sensor)'); axs[0].set_ylabel('share of mesh surface'); axs[0].legend(fontsize=7)
    axs[1].plot(cur['shifts'], cur['lidar_dist'], 'o-', color='tab:purple'); axs[1].axvline(0, color='gray', lw=0.6)
    axs[1].set_xlabel('shift along the LiDAR ray [m]'); axs[1].set_ylabel('median distance in-mask returns -> mesh [m]')
    fig.suptitle(f'{row.tag} #{row.id} {row.cls}', fontsize=9); plt.tight_layout()
    fig.savefig(RESULT_DIR / f'{row.tag}_obj{row.id}_control.png', dpi=80, bbox_inches='tight')
    if shown.get(row.tag, 0) < SHOW_OBJECT_FIGS:
        shown[row.tag] = shown.get(row.tag, 0) + 1
        plt.show()
    else:
        plt.close(fig)

### 7.1 Interactive 3D view

Drag to rotate, scroll to zoom, click legend entries to hide layers. Occupied voxels black, a random subset of the free voxels as green fog (there are far too many to draw all),
mesh surface coloured as in the 2D views, boxes as wireframes (orange = OBB of the mesh, green = GT box). Also saved as a standalone `.html` next to the figures.

In [ ]:
SHOW_3D = None            # None = the first object of every frame; or a list like [('ecp10_f560', 3), ('nusc3_kf3', 1)]
if SHOW_3D is None:
    SHOW_3D = [(tag, 1) for tag, c in CTX.items() if len(c['objs'])]
for tag, oid in SHOW_3D:
    c = CTX[tag]; r = c['objs'][oid - 1]; R = RES[(tag, oid)]
    gt = None if r['gt_idx'] is None else c['gts'][r['gt_idx']]
    boxes = [dict(corners=r['obb_corners'], color='orange', label='OBB of mesh')]
    if gt is not None:
        boxes.append(dict(corners=fv.box_corners(*pilot.gt_box_of(gt)), color='green', label='GT box'))
    fig3 = fv.scene3d(c['grid'], np.asarray(r['obb_center'], float), R['_surf'], boxes, half=max(6.0, 0.7 * max(r['obb_dims'][:2])),
                      d_min=D_CLAIM, title=f"{tag} #{oid} {r['prompt']}")
    fig3.write_html(str(RESULT_DIR / f"{tag}_obj{oid}_3d.html"))
    fig3.show()

## 8. Mask perturbation test (optional, GPU)

Re-run SAM3D Objects for chosen objects with only the mask changed (image, pointmap and seed identical; the pointmap stays the one built from the *original* mask,
with `PERTURB_FREEZE_SHIFT = True` SAM3D's depth anchor (shift and scale) is also still computed from the original mask, so the perturbation can only act through the crop, the image and the mask tokens; set it False to let the anchor follow the perturbed mask as it would in production, and compare both runs). Perturbation sizes are fractions of the object (erode / dilate radius = fraction of the shorter bounding-box side, shift = fraction of the box width): 5% is about SAM3D's training augmentation, 15-30% is far outside it and is what shows whether the mesh follows the mask. `repeat` changes nothing and is the reference (noise floor of a repeated run).

- **IoU-new / IoU-orig**: silhouette of the new mesh vs the mask SAM3D was given / vs the original SAM3 mask. If the mask were a hard constraint, IoU-new stays high and IoU-orig drops with the perturbation.
  If it is a hint, the mesh stays where the image says and IoU-new drops.
- **d_center, size ratios, d_yaw**: how much the mesh moved / resized versus the `repeat` run. **shift_dz**: change of SAM3D's depth anchor (the median 3D point in the mask); it is 0 when the shift is frozen, and tells you how much of a position change is just the anchor moving when it is not.
- **surf_free**: share of the mesh surface in free space (>= D_CLAIM), does it move with the mask?

In [ ]:
RUN_MASK_PERTURBATION = False       # set True on a GPU node
PERTURB_OBJECTS = None              # None = per frame the largest 'car' mask; or a list like [('nusc3_kf3', 1), ('ecp10_f560', 3)]
PERTURB_FREEZE_SHIFT = True         # True: SAM3D's depth anchor (shift/scale) stays computed from the ORIGINAL mask -> only the shape/crop effect is measured
PERTURB_VARIANTS = pilot.mask_variants(erode_dilate=(0.05, 0.15, 0.3), shift=(0.1, 0.3))   # fractions of the object's bounding box

if not RUN_MASK_PERTURBATION:
    print('Mask perturbation is off (RUN_MASK_PERTURBATION = False).')
else:
    if PERTURB_OBJECTS is None:
        PERTURB_OBJECTS = []
        for tag, c in CTX.items():
            cars = [(int(r['binary_mask'].sum()), i + 1) for i, r in enumerate(c['objs']) if r['prompt'] == 'car']
            if cars:
                PERTURB_OBJECTS.append((tag, max(cars)[1]))
    print('perturbing:', PERTURB_OBJECTS)
    PERT = []
    obj_model = None
    for tag, c in CTX.items():
        ids = [i for t_, i in PERTURB_OBJECTS if t_ == tag]
        if not ids:
            continue
        if obj_model is None:
            obj_model = pilot.build_objects_model(CFG[c['sel']['dataset']], DEVICE)
        sam3, pts_ng = pilot.load_cached_inputs(c['sel'], c['frame'], CKPT_ROOT)
        only = {(c['objs'][i - 1]['prompt'], c['objs'][i - 1]['sam3_index']) for i in ids}
        runs = pilot.run_mask_perturbation(obj_model, c['frame'], sam3, pts_ng, only, PERTURB_VARIANTS, freeze_shift=PERTURB_FREEZE_SHIFT)
        for row in pilot.perturbation_rows(runs, c['frame'], c['grid'], c['H'], c['W'], d_claim=D_CLAIM, device=DEVICE):
            row['tag'] = tag
            row['id'] = 1 + next(i for i, r in enumerate(c['objs']) if (r['prompt'], r['sam3_index']) == (row['prompt'], row['sam3_index']))
            PERT.append(row)
    if obj_model is not None:
        obj_model.unload(); gc.collect(); torch.cuda.empty_cache()
    dfp = pd.DataFrame(PERT)
    display(dfp.drop(columns=['prompt', 'sam3_index']))

In [ ]:
if RUN_MASK_PERTURBATION and len(dfp):
    for (tag, oid), g in dfp.groupby(['tag', 'id']):
        fig, axs = plt.subplots(1, 3, figsize=(17, 3.8)); x = np.arange(len(g))
        axs[0].plot(x, g.iou_new, 'o-', label='IoU with the mask SAM3D was given'); axs[0].plot(x, g.iou_orig, 's--', label='IoU with the ORIGINAL mask')
        axs[0].set_ylim(0, 1.02); axs[0].legend(fontsize=7)
        axs[1].plot(x, g.len_ratio, 'o-', label='length'); axs[1].plot(x, g.wid_ratio, 's-', label='width'); axs[1].plot(x, g.hgt_ratio, '^-', label='height')
        axs[1].axhline(1, color='k', lw=0.5); axs[1].legend(fontsize=7); axs[1].set_ylabel('size / size in the repeat run')
        axs[2].bar(x - 0.2, g.d_center, 0.4, label='centre shift [m]'); axs[2].bar(x + 0.2, g.shift_dz.abs(), 0.4, label='|shift_dz| [m]'); axs[2].legend(fontsize=7)
        for ax in axs:
            ax.set_xticks(x); ax.set_xticklabels(g.variant, rotation=45, ha='right', fontsize=8)
        fig.suptitle(f'{tag} #{oid}: effect of changing only the mask', fontsize=10); plt.tight_layout()
        fig.savefig(RESULT_DIR / f'perturb_{tag}_obj{oid}.png', dpi=90, bbox_inches='tight'); plt.show()

## 9. Summary across all frames (to paste / compare)

One row per object with the numbers that matter, and medians per frame over the objects the LiDAR actually observed (`surf_unknown` < 0.95). Everything is also saved as CSV.

In [ ]:
def dc(name):
    return df[name] if name in df else pd.Series(np.nan, index=df.index)
S = df[['tag', 'id', 'cls', 'has_gt', 'motion']].copy()
S['recall'] = dc('mask_recall'); S['iou'] = dc('mask_iou'); S['leak_free'] = dc('mask_leak_free'); S['border'] = dc('mask_touches_border')
S['depth_med'] = dc('depth_median')
S['surf_free'] = dc(f'surf_frac_free_m{D_CLAIM:g}'); S['surf_unk'] = dc('surf_frac_unknown'); S['pen_p50'] = dc('surf_viol_depth_p50')
S['floor'] = dc('floor_free'); S['floor_sc'] = dc('floor_scaled_free')
S['sh_free'] = dc('best_shift_free'); S['free@sh'] = dc('free_at_best_shift'); S['sh_lidar'] = dc('best_shift_lidar')
S['implied'] = dc('red_implied'); S['no_ret'] = dc('red_in_mask_no_return_near'); S['ret_nb'] = dc('red_in_mask_returns_not_behind'); S['outside'] = dc('red_outside_mask'); S['end_in_mesh'] = dc('red_end_in_mesh')
S['below_gt'] = dc('mesh_below_gt_bottom'); S['red_n'] = dc('red_n')
S = S.round(3)
display(S)
print(S.to_string(index=False))
obs = S[S.surf_unk < 0.95]
print()
print('medians over objects the LiDAR observed (surf_unk < 0.95), per frame:')
display(obs.groupby('tag')[['recall', 'iou', 'leak_free', 'depth_med', 'surf_free', 'floor', 'floor_sc', 'sh_free', 'sh_lidar', 'implied', 'no_ret', 'outside', 'end_in_mesh', 'below_gt']].median().round(3))

## 10. Save

In [ ]:
suffix = '_'.join(RUN_TAGS) if len(RUN_TAGS) < 3 else 'all_frames'
df.to_csv(RESULT_DIR / f'objects_{suffix}.csv', index=False)
T.to_csv(RESULT_DIR / f'freespace_table_{suffix}.csv', index=False)
S.to_csv(RESULT_DIR / f'summary_{suffix}.csv', index=False)
if RUN_MASK_PERTURBATION and len(dfp):
    dfp.to_csv(RESULT_DIR / f'mask_perturbation_{suffix}.csv', index=False)
print('saved to', RESULT_DIR)